In [1]:
%load_ext autoreload
%autoreload 2

from Utils import Notebook, tex, gph, sgn, nm, ds
import numpy as np
import scipy.linalg as la

from IPython.display import display, Math, Latex,Markdown

import ControllerDesigner

Notebook.setup()

LaTeX has been enabled for text rendering.


### Definição da Planta

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la
from IPython.display import display, Math, Markdown

A = np.array([[0.0, 1.0], [3.75, 0.0]], dtype=np.float64)
B = np.array([[0.0], [0.25]], dtype=np.float64)

h = 0.1
lambd = 1.0
upsilon = 0.1

design_params_ctrl = {'h': h, 'υ': upsilon, 'λ': lambd, 'A': A, 'B': B}
synthesis_results = ControllerDesigner.synthesize_setm(design_params_ctrl)

if synthesis_results is None:
  display(Markdown(
      "**Erro:** A síntese do Controlador PETC/SETM resultou em *Infeasible*."))
else:
  Xi = synthesis_results['etm']['Ξ']
  Psi = synthesis_results['etm']['Ψ']
  K = synthesis_results['controller']['K']
  P = synthesis_results['functional']['P']
  # opt_gamma_L = synthesis_results['optimal_value']['gamma_L']
  # opt_etm_matrices = synthesis_results['optimal_value']['etm_matrices']

  display(Markdown("#### 1. Parâmetros do Delineamento"))
  display(Math(
      rf'h = {h}\,\text{{s}} \quad \lambda = {lambd} \quad \upsilon = {upsilon}'
  ))

  display(Markdown("#### 2. Dinâmica do Sistema (Tempo Contínuo e Discreto)"))
  display(Math(rf'A = {tex.mat2tex(A)}, \quad B = {tex.mat2tex(B)}'))

  display(Markdown("#### 3. Matrizes de Ponderação do ETM Estático (SETM)"))
  display(Math(rf'\Xi = {tex.mat2tex(Xi)}'))
  display(Math(rf'\Psi = {tex.mat2tex(Psi)}'))

  # display(
  #     Markdown(f"**Valor Ótimo do Problema:** \
  #       `gamma={opt_gamma_L:.6f}`, `etm_matrices={opt_etm_matrices:.6f}`"))

  display(Markdown("#### 4. Ganhos de Controle e Estimação"))
  display(Math(rf'P = {tex.mat2tex(P)}'))
  display(Math(rf'K = {tex.mat2tex(K)}'))

cond(X) = 14.461147083701736
eig(X) = [9.85816892e-06 1.18926561e-04]
||X @ Xinv - I|| = 1.950151031221766e-16


#### 1. Parâmetros do Delineamento

<IPython.core.display.Math object>

#### 2. Dinâmica do Sistema (Tempo Contínuo e Discreto)

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

#### 4. Ganhos de Controle e Estimação

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
params_obs = {**design_params_ctrl, 'nu_bar': 1.0,
              'alpha': 0.5, 'rho': 0.01, 'C': np.eye(A.shape[0])}
decay_factor = params_obs['rho'] * \
    np.exp(2.0 * params_obs['alpha'] * params_obs['nu_bar'])

if decay_factor >= 1.0:
  print(f"Fator de decaimento do observador: {decay_factor:.4f}")
  display(Markdown(
      "**Aviso:** O fator de decaimento do observador é maior ou igual a 1.0, o que pode indicar instabilidade."
  ))
else:
  observer_results = ControllerDesigner.synthesize_impulsive_observer(
      params_obs)
  if observer_results is not None:
    L = observer_results["L"]
    display(Math(rf'L = {tex.mat2tex(L)}'))
  else:
    display(Markdown("A síntese do Observador resultou em \
                      *Infeasible* para os parâmetros dados."))

<IPython.core.display.Math object>

In [7]:
import numpy as np


def format_matrix_to_cpp(mat, name="X", scientific=True):
  """
  Converte um vetor ou matriz numpy para o formato C++:
  X = {{..., ...}, {..., ...}}
  """
  mat = np.atleast_2d(mat)
  rows, cols = mat.shape

  fmt = "{:.2e}" if scientific else "{:.4f}"

  rows_str = []
  for i in range(rows):
    row_vals = ", ".join([fmt.format(val) for val in mat[i]])
    rows_str.append(f"{row_vals}")

  inner_str = ", ".join(rows_str)
  return f"{name} = {{{inner_str}}};"


# Extração das variáveis do seu escopo
Xi = synthesis_results['etm']['Ξ']
Psi = synthesis_results['etm']['Ψ']
K = synthesis_results['controller']['K']
L = observer_results["L"]

# Exibição no console / Jupyter no formato exato solicitado
print(format_matrix_to_cpp(Xi, name="Xi"))
print(format_matrix_to_cpp(Psi, name="Psi"))
print(format_matrix_to_cpp(K, name="K"))
print(format_matrix_to_cpp(L, name="L"))

Xi = {1.07e+06, 5.09e+05, 5.09e+05, 2.41e+05};
Psi = {3.77e+04, 1.68e+04, 1.68e+04, 1.85e+04};
K = {-3.73e+01, -1.77e+01};
L = {6.60e-01, -4.52e-03, -1.02e-02, 6.73e-01};
